In [ ]:
import time
import subprocess
import sys

## `dcm2folder.py` – Reorganizing DICOM files by patient and view

**Algorithm description**

- Recursively scans the `data/raw` directory to find all DICOM files.
- For each file:
  - Reads the DICOM header (using `pydicom`) to obtain `PatientID`, laterality and view.
  - Builds the target path `data/processed/<ID>/dcm`.
  - Moves/renames the file to the corresponding folder.

**Time complexity**

Let \(N\) be the number of DICOM files:

- Reading the header: assumed \(O(1)\) (fixed-size metadata).
- Building paths and simple string operations: \(O(1)\).
- Moving/renaming the file: proportional to file size, which we assume bounded.

Therefore, overall time grows linearly with the number of files:

\[
T(N) = O(N)
\]

**Space complexity**

- The script processes one file at a time and does **not** keep all images in memory.
- It only stores a constant amount of metadata and paths.

\[
S(N) = O(1)
\]


In [ ]:
script_path = "dcm2folder.py"

start = time.perf_counter()

subprocess.run([sys.executable, script_path], check=True)

end = time.perf_counter()
print(f"Tiempo total de ejecución: {end - start:.6f} segundos")

## `dcm2png.py` – Converting DICOM to PNG

**Algorithm description**

- Iterates over patient folders in `data/processed` to locate DICOM files.
- For each DICOM:
  - Reads the DICOM pixel array using `pydicom`.
  - Converts the NumPy array to a `PIL.Image`.
  - Saves the image as PNG in the `img` subfolder.

**Time complexity**

Let:

- \(N\): number of images,
- \(W, H\): image width and height,
- \(P = W \cdot H\): number of pixels per image.

For each image:

- Read pixel data: \(O(P)\).
- Convert and save as PNG: \(O(P)\).

Total:

\[
T(N, P) = O(N \cdot P)
\]

**Space complexity**

- At any time, only one image array of size \(P\) is kept in memory (plus small overhead).

\[
S(P) = O(P)
\]


In [ ]:
script_path = "dcm2png.py"

start = time.perf_counter()

subprocess.run([sys.executable, script_path], check=True)

end = time.perf_counter()
print(f"Tiempo total de ejecución: {end - start:.6f} segundos")

## `dmctotiff.py` – Converting DICOM to TIFF

**Algorithm description**

- Similar pipeline to `dcm2png.py`, but saves the images in TIFF format:
  - Reads each DICOM from `data/processed`.
  - Extracts the pixel array.
  - Converts to `PIL.Image`.
  - Saves as TIFF in the `img` subfolder.

**Time complexity**

Using the same notation:

- \(N\): number of images,
- \(P = W $\cdot$ H\): pixels per image.

For each image:

- Read, convert and save: \(O(P)\).

Total:

\[
T(N, P) = O(N $\cdot$ P)
\]

**Space complexity**

- Stores one image array of size \(P\) at a time.

\[
S(P) = O(P)
\]


In [ ]:
script_path = "dcmtotiff.py"

start = time.perf_counter()

subprocess.run([sys.executable, script_path], check=True)

end = time.perf_counter()
print(f"Tiempo total de ejecución: {end - start:.6f} segundos")

## `segjson.py` – Organizing JSON segmentation files

**Algorithm description**

- Builds a dictionary mapping `PatientID` → patient folder in `data/processed`.
- Iterates over JSON segmentation files in `data/raw/TOMPEI-CMMD-Segmentations`.
- For each JSON:
  - Parses the filename (via regex) to extract ID, view and laterality.
  - Looks up the corresponding patient folder in the dictionary.
  - Creates the `seg/json` subfolder if it does not exist.
  - Copies/renames the JSON file to a standardized name.

**Time complexity**

Let:

- \(K\): number of patient folders,
- \(J\): number of JSON files.

Steps:

- Build the patient dictionary: \(O(K)\).
- For each JSON:
  - Regex parsing + dictionary lookup: \(O(1)\) average.
  - Copy of the JSON file: proportional to file size (assumed bounded).

Total:

\[
T(K, J) = O(K + J)
\]

**Space complexity**

- The main extra structure is the patient dictionary:

\[
S(K) = O(K)
\]


In [ ]:
script_path = "segjson.py"

start = time.perf_counter()

subprocess.run([sys.executable, script_path], check=True)

end = time.perf_counter()
print(f"Tiempo total de ejecución: {end - start:.6f} segundos")

## `segmask.py` – Generating binary masks from JSON segmentations

**Algorithm description**

- Defines a function `create_mask_from_json(json_path, size_wh)`:
  - Loads a JSON file containing one or more polygons.
  - Creates a blank mask image of size \(W \times H\).
  - Fills each polygon region in the mask (label 255).
- Iterates over all patients:
  - For each JSON file in `seg/json`, finds the corresponding image size.
  - Calls `create_mask_from_json` and saves the mask in `seg/mask`.

**Time complexity**

Let:

- \(P = W \cdot H\): number of pixels in the mask,
- \(V\): total number of polygon vertices in one JSON,
- \(J\): number of JSON files.

For a single mask:

- Creating the blank mask: \(O(P)\).
- Filling polygons: cost proportional to the number of pixels being painted; in the worst case, also \(O(P)\).

Thus, per JSON:

\[
T = O(P)
\]

For all JSON files:

\[
T(J, P) = O(J $\cdot$ P)
\]

**Space complexity**

- One mask array of size \(P\),
- Plus storage of the polygon vertices.

\[
S(P, V) = O(P + V)
\]


In [ ]:
script_path = "segmask.py"

start = time.perf_counter()

subprocess.run([sys.executable, script_path], check=True)

end = time.perf_counter()
print(f"Tiempo total de ejecución: {end - start:.6f} segundos")

In [ ]:
script_path = "segvis.py"

start = time.perf_counter()

subprocess.run([sys.executable, script_path], check=True)

end = time.perf_counter()
print(f"Tiempo total de ejecución: {end - start:.6f} segundos")